# Ungraded Lab: Walkthrough of ML Metadata

Keeping records at each stage of the project is an important aspect of machine learning pipelines. Especially in production models which involve many iterations of datasets and re-training, having these records will help in maintaining or debugging the deployed system. [ML Metadata](https://www.tensorflow.org/tfx/guide/mlmd) addresses this need by having an API suited specifically for keeping track of any progress made in ML projects.

As mentioned in earlier labs, you have already used ML Metadata when you ran your TFX pipelines. Each component automatically records information to a metadata store as you go through each stage. It allowed you to retrieve information such as the name of the training splits or the location of an inferred schema. 

In this notebook, you will look more closely at how ML Metadata can be used directly for recording and retrieving metadata independent from a TFX pipeline (i.e. without using TFX components). You will use TFDV to infer a schema and record all information about this process. These will show how the different components are related to each other so you can better interact with the database when you go back to using TFX in the next labs. Moreover, knowing the inner workings of the library will help you adapt it for other platforms if needed.

Let's get to it!

## Imports

In [ ]:
from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

import tensorflow as tf
print('TF version: {}'.format(tf.__version__))

import tensorflow_data_validation as tfdv
print('TFDV version: {}'.format(tfdv.version.__version__))

import pandas as pd
import os

## Download dataset

You will be using the [UCI Heart Disease](https://archive.ics.uci.edu/ml/datasets/Heart+Disease) dataset (Cleveland subset) for this lab. The dataset contains 303 patient records with 13 clinical features and a binary target indicating the presence of heart disease. We'll download the data, clean it, and split it into train/eval/serving sets.

In [ ]:
# Define column names for the UCI Heart Disease dataset
column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

try:
    # Download from UCI Machine Learning Repository
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
    data = pd.read_csv(url, header=None, names=column_names, na_values='?')

    # Drop rows with missing values (6 rows have '?' in ca and thal)
    data = data.dropna()

    # Binarize target: 0 = no disease, 1 = disease present
    data['target'] = (data['target'] > 0).astype(int)

    # Convert numeric columns to proper int types
    int_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
                'thalach', 'exang', 'slope', 'ca', 'thal', 'target']
    for col in int_cols:
        data[col] = data[col].astype(int)

    # Split: 60% train, 20% eval, 20% serving
    from sklearn.model_selection import train_test_split
    train, remainder = train_test_split(data, train_size=0.6, random_state=42)
    eval_data, serving = train_test_split(remainder, test_size=0.5, random_state=42)

    # Save splits to CSV
    for split_name, split_df in [('train', train), ('eval', eval_data), ('serving', serving)]:
        split_path = os.path.join('data', split_name)
        os.makedirs(split_path, exist_ok=True)
        split_df.to_csv(os.path.join(split_path, 'data.csv'), index=False)

    print(f"Downloaded and split Heart Disease dataset: {len(train)} train, {len(eval_data)} eval, {len(serving)} serving")

except Exception as e:
    print(f"Could not download dataset ({e}). Using existing local files.")

print("\nHere's what we have:")
!ls -R data

## Process Outline

Here is the figure shown in class that describes the different components in an ML Metadata store:

<img src='img/mlmd_overview.png' alt='image of mlmd overview'>

The green box in the middle shows the data model followed by ML Metadata. The [official documentation](https://www.tensorflow.org/tfx/guide/mlmd#data_model) describe each of these and we'll show it here as well for easy reference:

* `ArtifactType` describes an artifact's type and its properties that are stored in the metadata store. You can register these types on-the-fly with the metadata store in code, or you can load them in the store from a serialized format. Once you register a type, its definition is available throughout the lifetime of the store.
* An `Artifact` describes a specific instance of an ArtifactType, and its properties that are written to the metadata store.
* An `ExecutionType` describes a type of component or step in a workflow, and its runtime parameters.
* An `Execution` is a record of a component run or a step in an ML workflow and the runtime parameters. An execution can be thought of as an instance of an ExecutionType. Executions are recorded when you run an ML pipeline or step.
* An `Event` is a record of the relationship between artifacts and executions. When an execution happens, events record every artifact that was used by the execution, and every artifact that was produced. These records allow for lineage tracking throughout a workflow. By looking at all events, MLMD knows what executions happened and what artifacts were created as a result. MLMD can then recurse back from any artifact to all of its upstream inputs.
* A `ContextType` describes a type of conceptual group of artifacts and executions in a workflow, and its structural properties. For example: projects, pipeline runs, experiments, owners etc.
* A `Context` is an instance of a ContextType. It captures the shared information within the group. For example: project name, changelist commit id, experiment annotations etc. It has a user-defined unique name within its ContextType.
* An `Attribution` is a record of the relationship between artifacts and contexts.
* An `Association` is a record of the relationship between executions and contexts.

As mentioned earlier, you will use TFDV to generate a schema and record this process in the ML Metadata store. You will be starting from scratch so you will be defining each component of the data model. The outline of steps involve:

1. Defining the ML Metadata's storage database
1. Setting up the necessary artifact types
1. Setting up the execution types
1. Generating an input artifact unit
1. Generating an execution unit
1. Registering an input event
1. Running the TFDV component
1. Generating an output artifact unit
1. Registering an output event
1. Updating the execution unit
1. Seting up and generating a context unit
1. Generating attributions and associations

You can then retrieve information from the database to investigate aspects of your project. For example, you can find which dataset was used to generate a particular schema. You will also do that in this exercise.

For each of these steps, you may want to have the [MetadataStore API documentation](https://www.tensorflow.org/tfx/ml_metadata/api_docs/python/mlmd/MetadataStore) open so you can lookup any of the methods you will be using to interact with the metadata store. You can also look at the `metadata_store` protocol buffer [here](https://github.com/google/ml-metadata/blob/r0.24.0/ml_metadata/proto/metadata_store.proto) to see descriptions of each data type covered in this tutorial.

## Define ML Metadata's Storage Database

The first step would be to instantiate your storage backend. There are several types supported such as fake (temporary) database, SQLite, MySQL, and even cloud-based storage. Here, you will use a **SQLite database** so that the metadata persists across sessions. This means you can close the notebook and later query the metadata store to retrieve all recorded artifacts, executions, and lineage information.

In [ ]:
# Instantiate a connection config with a SQLite backend for persistent storage
connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = './metadata/mlmd.sqlite'

# Ensure the directory exists
os.makedirs('./metadata', exist_ok=True)

# Remove existing DB to start fresh each run (optional — remove this line to accumulate across runs)
if os.path.exists('./metadata/mlmd.sqlite'):
    os.remove('./metadata/mlmd.sqlite')

# Setup the metadata store
store = metadata_store.MetadataStore(connection_config)

print(f"Metadata store created at: ./metadata/mlmd.sqlite")

## Register ArtifactTypes

Next, you will create the artifact types needed and register them to the store. Since our simple exercise will just involve generating a schema using TFDV, you will only create two artifact types: one for the **input dataset** and another for the **output schema**. The main steps will be to:

* Declare an `ArtifactType()`
* Define the name of the artifact type
* Define the necessary properties within these artifact types. For example, it is important to know the data split name so you may want to have a `split` property for the artifact type that holds datasets.
* Use `put_artifact_type()` to register them to the metadata store. This generates an `id` that you can use later to refer to a particular artifact type.

*Bonus: For practice, you can also extend the code below to create an artifact type for the statistics.*

In [4]:
#Create ArtifactType for statistics
statistics_artifact_type = metadata_store_pb2.ArtifactType()
statistics_artifact_type.name = 'statistics'
statistics_artifact_type.properties['name'] = metadata_store_pb2.STRING
statistics_artifact_type.properties['split'] = metadata_store_pb2.STRING
statistics_artifact_type.properties['version'] = metadata_store_pb2.STRING

# Register artifact type to the Metadata Store
statistics_artifact_type_id = store.put_artifact_type(statistics_artifact_type)

In [ ]:
# Create ArtifactType for the input dataset
data_artifact_type = metadata_store_pb2.ArtifactType()
data_artifact_type.name = 'DataSet'
data_artifact_type.properties['name'] = metadata_store_pb2.STRING
data_artifact_type.properties['split'] = metadata_store_pb2.STRING
data_artifact_type.properties['version'] = metadata_store_pb2.INT

# Register artifact type to the Metadata Store
data_artifact_type_id = store.put_artifact_type(data_artifact_type)

# Create ArtifactType for Schema
schema_artifact_type = metadata_store_pb2.ArtifactType()
schema_artifact_type.name = 'Schema'
schema_artifact_type.properties['name'] = metadata_store_pb2.STRING
schema_artifact_type.properties['version'] = metadata_store_pb2.INT

# Register artifact type to the Metadata Store
schema_artifact_type_id = store.put_artifact_type(schema_artifact_type)

# Create ArtifactType for Anomalies (detected by TFDV)
anomaly_artifact_type = metadata_store_pb2.ArtifactType()
anomaly_artifact_type.name = 'Anomalies'
anomaly_artifact_type.properties['name'] = metadata_store_pb2.STRING
anomaly_artifact_type.properties['num_anomalies'] = metadata_store_pb2.INT
anomaly_artifact_type.properties['description'] = metadata_store_pb2.STRING

# Register artifact type to the Metadata Store
anomaly_artifact_type_id = store.put_artifact_type(anomaly_artifact_type)

# Create ArtifactType for a trained Model
model_artifact_type = metadata_store_pb2.ArtifactType()
model_artifact_type.name = 'Model'
model_artifact_type.properties['name'] = metadata_store_pb2.STRING
model_artifact_type.properties['version'] = metadata_store_pb2.INT
model_artifact_type.properties['framework'] = metadata_store_pb2.STRING

# Register artifact type to the Metadata Store
model_artifact_type_id = store.put_artifact_type(model_artifact_type)

# Create ArtifactType for Model Evaluation metrics
eval_artifact_type = metadata_store_pb2.ArtifactType()
eval_artifact_type.name = 'ModelEvaluation'
eval_artifact_type.properties['name'] = metadata_store_pb2.STRING
eval_artifact_type.properties['accuracy'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['f1_score'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['precision'] = metadata_store_pb2.DOUBLE
eval_artifact_type.properties['recall'] = metadata_store_pb2.DOUBLE

# Register artifact type to the Metadata Store
eval_artifact_type_id = store.put_artifact_type(eval_artifact_type)

print('Data artifact type:\n', data_artifact_type)
print('Schema artifact type:\n', schema_artifact_type)
print('Anomaly artifact type:\n', anomaly_artifact_type)
print('Statistics artifact type:\n', statistics_artifact_type)
print('Model artifact type:\n', model_artifact_type)
print('ModelEvaluation artifact type:\n', eval_artifact_type)
print('Data artifact type ID:', data_artifact_type_id)
print('Schema artifact type ID:', schema_artifact_type_id)
print('Anomaly artifact type ID:', anomaly_artifact_type_id)
print('Statistics artifact type ID:', statistics_artifact_type_id)
print('Model artifact type ID:', model_artifact_type_id)
print('ModelEvaluation artifact type ID:', eval_artifact_type_id)

## Register ExecutionType

You will then create the execution types needed. For the simple setup, you will just declare one for the data validation component with a `state` property so you can record if the process is running or already completed.

In [ ]:
# Create ExecutionType for Data Validation component
dv_execution_type = metadata_store_pb2.ExecutionType()
dv_execution_type.name = 'Data Validation'
dv_execution_type.properties['state'] = metadata_store_pb2.STRING

# Register execution type to the Metadata Store
dv_execution_type_id = store.put_execution_type(dv_execution_type)

# Create ExecutionType for Anomaly Detection component
anomaly_execution_type = metadata_store_pb2.ExecutionType()
anomaly_execution_type.name = 'Anomaly Detection'
anomaly_execution_type.properties['state'] = metadata_store_pb2.STRING

# Register execution type to the Metadata Store
anomaly_execution_type_id = store.put_execution_type(anomaly_execution_type)

# Create ExecutionType for Model Training component
training_execution_type = metadata_store_pb2.ExecutionType()
training_execution_type.name = 'Model Training'
training_execution_type.properties['state'] = metadata_store_pb2.STRING

# Register execution type to the Metadata Store
training_execution_type_id = store.put_execution_type(training_execution_type)

# Create ExecutionType for Model Evaluation component
eval_execution_type = metadata_store_pb2.ExecutionType()
eval_execution_type.name = 'Model Evaluation'
eval_execution_type.properties['state'] = metadata_store_pb2.STRING

# Register execution type to the Metadata Store
eval_execution_type_id = store.put_execution_type(eval_execution_type)

print('Data validation execution type:\n', dv_execution_type)
print('Data validation execution type ID:', dv_execution_type_id)
print('\nAnomaly detection execution type:\n', anomaly_execution_type)
print('Anomaly detection execution type ID:', anomaly_execution_type_id)
print('\nModel training execution type:\n', training_execution_type)
print('Model training execution type ID:', training_execution_type_id)
print('\nModel evaluation execution type:\n', eval_execution_type)
print('Model evaluation execution type ID:', eval_execution_type_id)

## Generate input artifact unit

With the artifact types created, you can now create instances of those types. The cell below creates the artifact for the input dataset. This artifact is recorded in the metadata store through the `put_artifacts()` function. Again, it generates an `id` that can be used for reference.

In [ ]:
# Declare input artifact of type DataSet
data_artifact = metadata_store_pb2.Artifact()
data_artifact.uri = './data/train/data.csv'
data_artifact.type_id = data_artifact_type_id
data_artifact.properties['name'].string_value = 'Heart Disease dataset'
data_artifact.properties['split'].string_value = 'train'
data_artifact.properties['version'].int_value = 1

# Submit input artifact to the Metadata Store
data_artifact_id = store.put_artifacts([data_artifact])[0]

print('Data artifact:\n', data_artifact)
print('Data artifact ID:', data_artifact_id)

## Generate execution unit

Next, you will create an instance of the `Data Validation` execution type you registered earlier. You will set the state to `RUNNING` to signify that you are about to run the TFDV function. This is recorded with the `put_executions()` function.

In [8]:
# Register the Execution of a Data Validation run
dv_execution = metadata_store_pb2.Execution()
dv_execution.type_id = dv_execution_type_id
dv_execution.properties['state'].string_value = 'RUNNING'

# Submit execution unit to the Metadata Store
dv_execution_id = store.put_executions([dv_execution])[0]

print('Data validation execution:\n', dv_execution)
print('Data validation execution ID:', dv_execution_id)

Data validation execution:
 type_id: 13
properties {
  key: "state"
  value {
    string_value: "RUNNING"
  }
}

Data validation execution ID: 1


## Register input event

An event defines a relationship between artifacts and executions. You will generate the input event relationship for dataset artifact and data validation execution units. The list of event types are shown [here](https://github.com/google/ml-metadata/blob/master/ml_metadata/proto/metadata_store.proto#L187) and the event is recorded with the `put_events()` function.

In [9]:
# Declare the input event
input_event = metadata_store_pb2.Event()
input_event.artifact_id = data_artifact_id
input_event.execution_id = dv_execution_id
input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

# Submit input event to the Metadata Store
store.put_events([input_event])

print('Input event:\n', input_event)

Input event:
 artifact_id: 1
execution_id: 1
type: DECLARED_INPUT



## Run the TFDV component

You will now run the TFDV component to generate the schema of dataset. This should look familiar since you've done this already in Week 1.

In [10]:
# Infer a schema by passing statistics to `infer_schema()`
train_data = './data/train/data.csv'
train_stats = tfdv.generate_statistics_from_csv(data_location=train_data)
schema = tfdv.infer_schema(statistics=train_stats)

schema_file = './schema.pbtxt'
tfdv.write_schema_text(schema, schema_file)

print("Dataset's Schema has been generated at:", schema_file)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Dataset's Schema has been generated at: ./schema.pbtxt


## Generate output artifact unit

Now that the TFDV component has finished running and schema has been generated, you can create the artifact for the generated schema.

In [ ]:
# Declare output artifact of type Schema_artifact
schema_artifact = metadata_store_pb2.Artifact()
schema_artifact.uri = schema_file
schema_artifact.type_id = schema_artifact_type_id
schema_artifact.properties['version'].int_value = 1
schema_artifact.properties['name'].string_value = 'Heart Disease Schema'

# Submit output artifact to the Metadata Store
schema_artifact_id = store.put_artifacts([schema_artifact])[0]

print('Schema artifact:\n', schema_artifact)
print('Schema artifact ID:', schema_artifact_id)

## Register output event

Analogous to the input event earlier, you also want to define an output event to record the ouput artifact of a particular execution unit.

In [12]:
# Declare the output event
output_event = metadata_store_pb2.Event()
output_event.artifact_id = schema_artifact_id
output_event.execution_id = dv_execution_id
output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

# Submit output event to the Metadata Store
store.put_events([output_event])

print('Output event:\n', output_event)

Output event:
 artifact_id: 2
execution_id: 1
type: DECLARED_OUTPUT



## Update the execution unit

As the TFDV component has finished running successfully, you need to update the `state` of the execution unit and record it again to the store.

In [13]:
# Mark the `state` as `COMPLETED`
dv_execution.id = dv_execution_id
dv_execution.properties['state'].string_value = 'COMPLETED'

# Update execution unit in the Metadata Store
store.put_executions([dv_execution])

print('Data validation execution:\n', dv_execution)

Data validation execution:
 id: 1
type_id: 13
properties {
  key: "state"
  value {
    string_value: "COMPLETED"
  }
}



## Anomaly Detection with MLMD Tracking

With a schema inferred from the training data, you can now use TFDV to check the eval data for anomalies — feature values or distributions that don't conform to the schema. This is a critical step in production ML pipelines to catch data drift or quality issues before training. You will record the anomaly detection process and its results as MLMD artifacts.

In [ ]:
# Create Anomaly Detection execution and set state to RUNNING
anomaly_execution = metadata_store_pb2.Execution()
anomaly_execution.type_id = anomaly_execution_type_id
anomaly_execution.properties['state'].string_value = 'RUNNING'

anomaly_execution_id = store.put_executions([anomaly_execution])[0]

# Register input events: the eval dataset and the schema are both inputs
eval_data_for_anomaly = metadata_store_pb2.Artifact()
eval_data_for_anomaly.uri = './data/eval/data.csv'
eval_data_for_anomaly.type_id = data_artifact_type_id
eval_data_for_anomaly.properties['name'].string_value = 'Heart Disease dataset'
eval_data_for_anomaly.properties['split'].string_value = 'eval'
eval_data_for_anomaly.properties['version'].int_value = 1

eval_data_for_anomaly_id = store.put_artifacts([eval_data_for_anomaly])[0]

# Link eval dataset as input
anomaly_eval_input = metadata_store_pb2.Event()
anomaly_eval_input.artifact_id = eval_data_for_anomaly_id
anomaly_eval_input.execution_id = anomaly_execution_id
anomaly_eval_input.type = metadata_store_pb2.Event.DECLARED_INPUT

# Link schema as input
anomaly_schema_input = metadata_store_pb2.Event()
anomaly_schema_input.artifact_id = schema_artifact_id
anomaly_schema_input.execution_id = anomaly_execution_id
anomaly_schema_input.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([anomaly_eval_input, anomaly_schema_input])

print('Anomaly detection execution ID:', anomaly_execution_id)
print('Eval data artifact ID (for anomaly detection):', eval_data_for_anomaly_id)
print('\nInput events registered: eval dataset + schema')

### Run TFDV anomaly detection

Use TFDV to generate statistics from the eval data and validate them against the inferred schema. Any discrepancies (e.g., unexpected feature values, missing features, schema mismatches) will be reported as anomalies.

In [ ]:
# Generate statistics for the eval data
eval_data_path = './data/eval/data.csv'
eval_stats = tfdv.generate_statistics_from_csv(data_location=eval_data_path)

# Validate eval statistics against the inferred schema
anomalies = tfdv.validate_statistics(statistics=eval_stats, schema=schema)

# Save anomalies report
anomalies_path = './anomalies.pbtxt'
from tensorflow_metadata.proto.v0 import anomalies_pb2
with open(anomalies_path, 'w') as f:
    f.write(str(anomalies))

# Count and summarize anomalies
anomaly_dict = dict(anomalies.anomaly_info)
num_anomalies = len(anomaly_dict)

if num_anomalies > 0:
    print(f"Found {num_anomalies} anomalies in eval data:")
    for feature_name, anomaly_info in anomaly_dict.items():
        print(f"  - {feature_name}: {anomaly_info.short_description}")
else:
    print("No anomalies found — eval data conforms to the schema.")

print(f"\nAnomalies report saved to: {anomalies_path}")

### Record anomaly results artifact and complete the execution

Create an `Anomalies` artifact to store the anomaly detection results. The number of anomalies found and a summary description are stored as MLMD properties for easy querying.

In [ ]:
# Build a description string summarizing anomalies
if num_anomalies > 0:
    anomaly_desc = '; '.join([f"{k}: {v.short_description}" for k, v in anomaly_dict.items()])
else:
    anomaly_desc = 'No anomalies detected'

# Create the Anomalies artifact
anomaly_artifact = metadata_store_pb2.Artifact()
anomaly_artifact.uri = anomalies_path
anomaly_artifact.type_id = anomaly_artifact_type_id
anomaly_artifact.properties['name'].string_value = 'Heart Disease Eval Anomalies'
anomaly_artifact.properties['num_anomalies'].int_value = num_anomalies
anomaly_artifact.properties['description'].string_value = anomaly_desc

anomaly_artifact_id = store.put_artifacts([anomaly_artifact])[0]

# Register output event linking anomalies to the execution
anomaly_output_event = metadata_store_pb2.Event()
anomaly_output_event.artifact_id = anomaly_artifact_id
anomaly_output_event.execution_id = anomaly_execution_id
anomaly_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([anomaly_output_event])

# Mark anomaly detection execution as COMPLETED
anomaly_execution.id = anomaly_execution_id
anomaly_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([anomaly_execution])

print('Anomaly artifact:\n', anomaly_artifact)
print('Anomaly artifact ID:', anomaly_artifact_id)
print('\nAnomaly detection execution state:', anomaly_execution.properties['state'].string_value)

## Model Training with MLMD Tracking

Now that data validation is complete, you will extend the pipeline by training a model and tracking it through MLMD. This mirrors what a real ML pipeline does: the training component consumes the validated dataset and produces a trained model artifact. You will follow the same pattern as before — create an execution, register input/output events, run the actual training, and record the output.

In [ ]:
# Create a Model Training execution and set state to RUNNING
training_execution = metadata_store_pb2.Execution()
training_execution.type_id = training_execution_type_id
training_execution.properties['state'].string_value = 'RUNNING'

# Submit execution unit to the Metadata Store
training_execution_id = store.put_executions([training_execution])[0]

# Register the dataset as input to the training execution
training_input_event = metadata_store_pb2.Event()
training_input_event.artifact_id = data_artifact_id
training_input_event.execution_id = training_execution_id
training_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([training_input_event])

print('Training execution:\n', training_execution)
print('Training execution ID:', training_execution_id)
print('\nTraining input event:\n', training_input_event)

### Train a RandomForest classifier

With the execution registered and input event recorded, you can now run the actual training. You will use scikit-learn's `RandomForestClassifier` to train a binary classifier on the Heart Disease dataset and save the model as a pickle file.

In [ ]:
import pickle
from sklearn.ensemble import RandomForestClassifier

# Load the training data
train_df = pd.read_csv('./data/train/data.csv')
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

# Train a RandomForest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save the trained model
model_path = './model/model.pkl'
os.makedirs('./model', exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model trained on {len(X_train)} samples with {X_train.shape[1]} features")
print(f"Training accuracy: {model.score(X_train, y_train):.4f}")
print(f"Model saved to: {model_path}")

### Record the trained model artifact and complete the execution

Now that training is done, you will create the model artifact, register it as an output of the training execution, and mark the execution as completed.

In [ ]:
# Create the trained model artifact
model_artifact = metadata_store_pb2.Artifact()
model_artifact.uri = model_path
model_artifact.type_id = model_artifact_type_id
model_artifact.properties['name'].string_value = 'Heart Disease RandomForest'
model_artifact.properties['version'].int_value = 1
model_artifact.properties['framework'].string_value = 'scikit-learn'

# Submit model artifact to the Metadata Store
model_artifact_id = store.put_artifacts([model_artifact])[0]

# Register output event linking model to training execution
training_output_event = metadata_store_pb2.Event()
training_output_event.artifact_id = model_artifact_id
training_output_event.execution_id = training_execution_id
training_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([training_output_event])

# Mark training execution as COMPLETED
training_execution.id = training_execution_id
training_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([training_execution])

print('Model artifact:\n', model_artifact)
print('Model artifact ID:', model_artifact_id)
print('\nTraining output event:\n', training_output_event)
print('\nTraining execution state:', training_execution.properties['state'].string_value)

## Model Evaluation with MLMD Tracking

With a trained model in hand, the next pipeline step is evaluation. You will evaluate the model on the held-out eval dataset and record the resulting metrics (accuracy, F1 score, precision, recall) as a `ModelEvaluation` artifact in MLMD. The evaluation execution takes both the model and eval dataset as inputs and produces the metrics artifact as output.

In [ ]:
# Register the eval dataset as a separate artifact (eval split)
eval_data_artifact = metadata_store_pb2.Artifact()
eval_data_artifact.uri = './data/eval/data.csv'
eval_data_artifact.type_id = data_artifact_type_id
eval_data_artifact.properties['name'].string_value = 'Heart Disease dataset'
eval_data_artifact.properties['split'].string_value = 'eval'
eval_data_artifact.properties['version'].int_value = 1

eval_data_artifact_id = store.put_artifacts([eval_data_artifact])[0]

# Create a Model Evaluation execution and set state to RUNNING
eval_execution = metadata_store_pb2.Execution()
eval_execution.type_id = eval_execution_type_id
eval_execution.properties['state'].string_value = 'RUNNING'

eval_execution_id = store.put_executions([eval_execution])[0]

# Register input events: the model and eval dataset are both inputs to evaluation
model_input_event = metadata_store_pb2.Event()
model_input_event.artifact_id = model_artifact_id
model_input_event.execution_id = eval_execution_id
model_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

eval_data_input_event = metadata_store_pb2.Event()
eval_data_input_event.artifact_id = eval_data_artifact_id
eval_data_input_event.execution_id = eval_execution_id
eval_data_input_event.type = metadata_store_pb2.Event.DECLARED_INPUT

store.put_events([model_input_event, eval_data_input_event])

print('Eval dataset artifact ID:', eval_data_artifact_id)
print('Evaluation execution ID:', eval_execution_id)
print('\nModel input event:\n', model_input_event)
print('Eval data input event:\n', eval_data_input_event)

### Run evaluation and compute metrics

Now run the actual evaluation by loading the eval dataset, generating predictions, and computing classification metrics.

In [ ]:
import json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load the eval data
eval_df = pd.read_csv('./data/eval/data.csv')
X_eval = eval_df.drop('target', axis=1)
y_eval = eval_df['target']

# Generate predictions
y_pred = model.predict(X_eval)

# Compute metrics
metrics = {
    'accuracy': accuracy_score(y_eval, y_pred),
    'f1_score': f1_score(y_eval, y_pred),
    'precision': precision_score(y_eval, y_pred),
    'recall': recall_score(y_eval, y_pred)
}

# Save metrics to a JSON file
metrics_path = './model/eval_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Evaluation on {len(X_eval)} samples:")
for metric_name, value in metrics.items():
    print(f"  {metric_name}: {value:.4f}")
print(f"\nMetrics saved to: {metrics_path}")

### Record the evaluation metrics artifact and complete the execution

Create a `ModelEvaluation` artifact that stores the computed metrics directly as MLMD properties. This allows you to query and compare metrics across different model versions through the metadata store.

In [ ]:
# Create the ModelEvaluation artifact with metrics as properties
eval_artifact = metadata_store_pb2.Artifact()
eval_artifact.uri = metrics_path
eval_artifact.type_id = eval_artifact_type_id
eval_artifact.properties['name'].string_value = 'Heart Disease RF Evaluation'
eval_artifact.properties['accuracy'].double_value = metrics['accuracy']
eval_artifact.properties['f1_score'].double_value = metrics['f1_score']
eval_artifact.properties['precision'].double_value = metrics['precision']
eval_artifact.properties['recall'].double_value = metrics['recall']

# Submit evaluation artifact to the Metadata Store
eval_artifact_id = store.put_artifacts([eval_artifact])[0]

# Register output event linking metrics to evaluation execution
eval_output_event = metadata_store_pb2.Event()
eval_output_event.artifact_id = eval_artifact_id
eval_output_event.execution_id = eval_execution_id
eval_output_event.type = metadata_store_pb2.Event.DECLARED_OUTPUT

store.put_events([eval_output_event])

# Mark evaluation execution as COMPLETED
eval_execution.id = eval_execution_id
eval_execution.properties['state'].string_value = 'COMPLETED'
store.put_executions([eval_execution])

print('Evaluation artifact:\n', eval_artifact)
print('Evaluation artifact ID:', eval_artifact_id)
print('\nEvaluation output event:\n', eval_output_event)
print('\nEvaluation execution state:', eval_execution.properties['state'].string_value)

## Setting up Context Types and Generating a Context Unit

You can group the artifacts and execution units into a `Context`. First, you need to define a `ContextType` which defines the required context. It follows a similar format as artifact and event types. You can register this with the `put_context_type()` function.

In [14]:
# Create a ContextType
expt_context_type = metadata_store_pb2.ContextType()
expt_context_type.name = 'Experiment'
expt_context_type.properties['note'] = metadata_store_pb2.STRING

# Register context type to the Metadata Store
expt_context_type_id = store.put_context_type(expt_context_type)

Similarly, you can create an instance of this context type and use the `put_contexts()` method to register to the store.

In [ ]:
# Generate the context
expt_context = metadata_store_pb2.Context()
expt_context.type_id = expt_context_type_id
# Give the experiment a name
expt_context.name = 'Heart Disease Pipeline'
expt_context.properties['note'].string_value = 'Data validation, anomaly detection, model training, and evaluation for UCI Heart Disease dataset'

# Submit context to the Metadata Store
expt_context_id = store.put_contexts([expt_context])[0]

print('Experiment Context type:\n', expt_context_type)
print('Experiment Context type ID: ', expt_context_type_id)

print('Experiment Context:\n', expt_context)
print('Experiment Context ID: ', expt_context_id)

## Generate attribution and association relationships

With the `Context` defined, you can now create its relationship with all artifacts and executions. You will create attributions linking the schema, anomalies, model, and evaluation artifacts to the experiment context, and associations linking all four execution steps to the context.

In [ ]:
# Generate attributions for schema, anomalies, model, and evaluation artifacts
schema_attribution = metadata_store_pb2.Attribution()
schema_attribution.artifact_id = schema_artifact_id
schema_attribution.context_id = expt_context_id

anomaly_attribution = metadata_store_pb2.Attribution()
anomaly_attribution.artifact_id = anomaly_artifact_id
anomaly_attribution.context_id = expt_context_id

model_attribution = metadata_store_pb2.Attribution()
model_attribution.artifact_id = model_artifact_id
model_attribution.context_id = expt_context_id

eval_attribution = metadata_store_pb2.Attribution()
eval_attribution.artifact_id = eval_artifact_id
eval_attribution.context_id = expt_context_id

# Generate associations for all four execution steps
dv_association = metadata_store_pb2.Association()
dv_association.execution_id = dv_execution_id
dv_association.context_id = expt_context_id

anomaly_association = metadata_store_pb2.Association()
anomaly_association.execution_id = anomaly_execution_id
anomaly_association.context_id = expt_context_id

training_association = metadata_store_pb2.Association()
training_association.execution_id = training_execution_id
training_association.context_id = expt_context_id

eval_association = metadata_store_pb2.Association()
eval_association.execution_id = eval_execution_id
eval_association.context_id = expt_context_id

# Submit all attributions and associations to the Metadata Store
store.put_attributions_and_associations(
    [schema_attribution, anomaly_attribution, model_attribution, eval_attribution],
    [dv_association, anomaly_association, training_association, eval_association]
)

print('Attributions: Schema, Anomalies, Model, Evaluation')
print('Associations: Data Validation, Anomaly Detection, Model Training, Model Evaluation')

## Retrieving Information from the Metadata Store

You've now recorded the needed information to the metadata store. If we did this in a persistent database, you can track which artifacts and events are related to each other even without seeing the code used to generate it. See a sample run below where you investigate what dataset is used to generate the schema. (**It would be obvious which dataset is used in our simple demo because we only have two artifacts registered. Thus, assume that you have thousands of entries in the metadata store.*)

In [17]:
# Get artifact types
store.get_artifact_types()

[id: 10
 name: "statistics"
 properties {
   key: "name"
   value: STRING
 }
 properties {
   key: "split"
   value: STRING
 }
 properties {
   key: "version"
   value: STRING
 },
 id: 11
 name: "DataSet"
 properties {
   key: "name"
   value: STRING
 }
 properties {
   key: "split"
   value: STRING
 }
 properties {
   key: "version"
   value: INT
 },
 id: 12
 name: "Schema"
 properties {
   key: "name"
   value: STRING
 }
 properties {
   key: "version"
   value: INT
 }]

In [19]:
# Get 1st element in the list of `Schema` artifacts.
# You will investigate which dataset was used to generate it.
schema_to_inv = store.get_artifacts_by_type('Schema')[0]

# print output
print(schema_to_inv)

id: 2
type_id: 12
uri: "./schema.pbtxt"
properties {
  key: "name"
  value {
    string_value: "Chicago Taxi Schema"
  }
}
properties {
  key: "version"
  value {
    int_value: 1
  }
}
type: "Schema"
create_time_since_epoch: 1740346178359
last_update_time_since_epoch: 1740346178359



In [20]:
# Get events related to the schema id
schema_events = store.get_events_by_artifact_ids([schema_to_inv.id])

print(schema_events)

[artifact_id: 2
execution_id: 1
type: DECLARED_OUTPUT
milliseconds_since_epoch: 1740346198967
]


You see that it is an output of an execution so you can look up the execution id to see related artifacts.

In [22]:
# Get events related to the output above
execution_events = store.get_events_by_execution_ids([schema_events[0].execution_id])

print(execution_events)

[artifact_id: 1
execution_id: 1
type: DECLARED_INPUT
milliseconds_since_epoch: 1740346125272
, artifact_id: 2
execution_id: 1
type: DECLARED_OUTPUT
milliseconds_since_epoch: 1740346198967
]


You see the declared input of this execution so you can select that from the list and lookup the details of the artifact.

In [23]:
# Look up the artifact that is a declared input
artifact_input = execution_events[0]

store.get_artifacts_by_id([artifact_input.artifact_id])

[id: 1
 type_id: 11
 uri: "./data/train/data.csv"
 properties {
   key: "name"
   value {
     string_value: "Chicago Taxi dataset"
   }
 }
 properties {
   key: "split"
   value {
     string_value: "train"
   }
 }
 properties {
   key: "version"
   value {
     int_value: 1
   }
 }
 type: "DataSet"
 create_time_since_epoch: 1740346092535
 last_update_time_since_epoch: 1740346092535]

### Retrieve anomaly detection results

You can also query the metadata store to check if any anomalies were found during data validation, and trace which schema and dataset were used for the check.

In [ ]:
# Get the anomaly detection results
anomaly_to_inv = store.get_artifacts_by_type('Anomalies')[0]
print(f"Anomalies found: {anomaly_to_inv.properties['num_anomalies'].int_value}")
print(f"Description: {anomaly_to_inv.properties['description'].string_value}")

# Trace back: Anomalies -> Anomaly Detection Execution -> Inputs (schema + eval dataset)
anomaly_events = store.get_events_by_artifact_ids([anomaly_to_inv.id])
anomaly_exec_events = store.get_events_by_execution_ids([anomaly_events[0].execution_id])

print('\nInputs to the anomaly detection execution:')
for event in anomaly_exec_events:
    if event.type == metadata_store_pb2.Event.DECLARED_INPUT:
        artifact = store.get_artifacts_by_id([event.artifact_id])[0]
        artifact_type = store.get_artifact_types_by_id([artifact.type_id])[0]
        print(f"  [{artifact_type.name}] {artifact.properties['name'].string_value} (uri: {artifact.uri})")

Now let's also trace the lineage of the trained model. You can investigate which dataset was used to train it by following the same event-based approach.

In [ ]:
# Get the trained model artifact
model_to_inv = store.get_artifacts_by_type('Model')[0]
print('Model artifact to investigate:\n', model_to_inv)

In [ ]:
# Trace the model's lineage: Model -> Training Execution -> Input Dataset
model_events = store.get_events_by_artifact_ids([model_to_inv.id])
print('Events for the model artifact:')
print(model_events)

# Get the training execution that produced this model
training_exec_id = model_events[0].execution_id
training_events = store.get_events_by_execution_ids([training_exec_id])
print('\nAll events for the training execution:')
print(training_events)

# Find the input artifact (dataset) used for training
training_input = [e for e in training_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT][0]
training_dataset = store.get_artifacts_by_id([training_input.artifact_id])[0]
print('\nDataset used to train the model:')
print(training_dataset)

You can see the full lineage: the model was produced by the Model Training execution, which consumed the Heart Disease dataset as input. This demonstrates how MLMD enables end-to-end provenance tracking across multiple pipeline steps.

### Trace evaluation metrics lineage

Finally, let's trace the evaluation metrics back to the model and dataset that produced them. This demonstrates end-to-end lineage across all three pipeline steps: data validation, training, and evaluation.

In [ ]:
# Get the evaluation metrics artifact
eval_to_inv = store.get_artifacts_by_type('ModelEvaluation')[0]
print('Evaluation artifact metrics:')
print(f"  Accuracy:  {eval_to_inv.properties['accuracy'].double_value:.4f}")
print(f"  F1 Score:  {eval_to_inv.properties['f1_score'].double_value:.4f}")
print(f"  Precision: {eval_to_inv.properties['precision'].double_value:.4f}")
print(f"  Recall:    {eval_to_inv.properties['recall'].double_value:.4f}")

# Trace back: Evaluation -> Execution -> Inputs (model + eval dataset)
eval_events = store.get_events_by_artifact_ids([eval_to_inv.id])
eval_exec_events = store.get_events_by_execution_ids([eval_events[0].execution_id])

print('\nInputs to the evaluation execution:')
for event in eval_exec_events:
    if event.type == metadata_store_pb2.Event.DECLARED_INPUT:
        artifact = store.get_artifacts_by_id([event.artifact_id])[0]
        artifact_type = store.get_artifact_types_by_id([artifact.type_id])[0]
        print(f"  [{artifact_type.name}] {artifact.properties['name'].string_value} (uri: {artifact.uri})")

In [ ]:
# Full pipeline summary: list all artifacts and executions in the experiment context
print('=== All artifacts in the experiment context ===')
context_artifacts = store.get_artifacts_by_context(expt_context_id)
for a in context_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f"  [{a_type.name}] {a.properties['name'].string_value} (id: {a.id})")

print('\n=== All executions in the experiment context ===')
context_executions = store.get_executions_by_context(expt_context_id)
for e in context_executions:
    e_type = store.get_execution_types_by_id([e.type_id])[0]
    print(f"  [{e_type.name}] state: {e.properties['state'].string_value} (id: {e.id})")

### Wrap Up

In this notebook, you practiced using ML Metadata outside of TFX to track a complete ML pipeline with four stages:

1. **Data Validation** — Inferred a schema from training data using TFDV
2. **Anomaly Detection** — Validated eval data against the schema to catch data drift
3. **Model Training** — Trained a RandomForest classifier on the Heart Disease dataset
4. **Model Evaluation** — Computed accuracy, F1, precision, and recall metrics

All artifacts, executions, and their relationships were recorded in a **persistent SQLite-backed** metadata store, enabling full lineage tracking across sessions. This should help you understand the inner workings of MLMD so you can better query metadata stores or adapt them for your own use cases.